# PlantDoc training

Set runtime to T4 GPU, then Run all.

In [ ]:
!pip install -q tensorflow
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
DATA_ROOT = Path('/content/PlantDoc-Dataset')
if not (DATA_ROOT / 'train').is_dir():
    !git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /content/PlantDoc-Dataset
train_dir = DATA_ROOT / 'train'
test_dir = DATA_ROOT / 'test'
print('Train exists:', train_dir.is_dir(), '| Test exists:', test_dir.is_dir())

In [ ]:
IMG_SIZE = (100, 100)
BATCH = 32

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    return tf.clip_by_value(img, 0, 1), label

train_class_dirs = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
test_class_dirs = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])
class_names = sorted(set(train_class_dirs) & set(test_class_dirs))
num_classes = len(class_names)
print('Classes in both train & test:', num_classes, '| Train-only:', set(train_class_dirs) - set(test_class_dirs))

train_ds_raw = keras.utils.image_dataset_from_directory(
    train_dir, labels='inferred', label_mode='categorical',
    image_size=IMG_SIZE, batch_size=BATCH, shuffle=True, seed=42,
    class_names=class_names,
)
train_ds = train_ds_raw.map(lambda x,y: (keras.layers.Rescaling(1/255.)(x), y), num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir, labels='inferred', label_mode='categorical',
    image_size=IMG_SIZE, batch_size=BATCH, shuffle=False,
    class_names=class_names,
)
test_ds = test_ds.map(lambda x,y: (keras.layers.Rescaling(1/255.)(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [ ]:
MODEL_NAME = 'VGG16'
inp = keras.layers.Input(shape=(*IMG_SIZE, 3))
if MODEL_NAME == 'VGG16':
    base = keras.applications.VGG16(include_top=False, weights='imagenet', input_tensor=inp, pooling='avg')
elif MODEL_NAME == 'InceptionV3':
    base = keras.applications.InceptionV3(include_top=False, weights='imagenet', input_tensor=inp, pooling='avg')
else:
    base = keras.applications.InceptionResNetV2(include_top=False, weights='imagenet', input_tensor=inp, pooling='avg')
out = keras.layers.Dense(num_classes, activation='softmax')(base.output)
model = keras.Model(inp, out)
model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
ckpt_path = f'/content/{MODEL_NAME.lower()}_plantdoc.keras'
model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[
        keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor='val_accuracy', mode='max'),
    ],
)
print('Training done. Model path:', ckpt_path)

In [ ]:
import json
with open('/content/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

Export weights for the local API. Run the next cell and download the .weights.h5 file.

In [ ]:
weights_path = f'/content/{MODEL_NAME.lower()}_plantdoc.weights.h5'
model.save_weights(weights_path)
from google.colab import files
files.download(weights_path)

In [ ]:
from google.colab import files
files.download(ckpt_path)
files.download('/content/class_names.json')